# Loading the saved models

The model notebooks save what they train (`utils/saving.py`); this notebook shows how to load each kind of saved model again and use it **without training anything**: a forecast for the 24 hours after the last hour of the data, and a check on the last 30 days.

| Type | Saved by | File | Loaded with | Also needed |
|---|---|---|---|---|
| 1. One scikit-learn model (Ridge, Gradient Boosting, Random Forest, pipelines...) | `classical-models`, `neural-models`, `data-analysis` | `models/<notebook>/<file>.joblib` | `load_model(notebook, name)` | the input columns, built as in the notebook (`inputs` entry) |
| 2. A dict or list of scikit-learn objects (quantile models, K-Means + scaler, MLP seeds) | the same | `.joblib` | `load_model` | the same |
| 3. PyTorch networks: one `state_dict` per seed | `neural-models` | `.pt` | `load_model` → `net.load_state_dict(state)` | the network class, its setting (in the name) and the scalings of the inputs and the target |
| 4. The Timestral export: Gradient Boosting + the Timestral setting | `timestral-zero-shot` §3.6 | `models/<id>/models.joblib` + `metadata.json` | `joblib.load` | the Timestral weights (Hugging Face) for the zero-shot model |

`models/<notebook>/index.json` lists every saved model with its name, file, save time and extra information (number of epochs, seeds, scale of the target). The names are the ones of the notebooks' tables, prefixed by the question: e.g. `hourly |r|: LSTM {'hidden_size': 64}`.

**Before running:** run the model notebooks with `SAVE_MODELS = True` (the default). Each section below is skipped, with a message, when its models are not in `models/`. Run from the repo root.

## 1. Settings and the saved models

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # conda's MKL numpy and pip's torch each ship libiomp5md.dll (model_test.ipynb)
import ast
import json
import re
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

sys.path.insert(0, os.getcwd())   # utils/ is in the repo root
from utils import hourly as U
from utils.config import DATA_START, TEST_START
from utils.features import (ORIGIN_FEATURES, TARGET_HOUR_FEATURES, load_releases, next_day_release_features,
                            origin_features, recent_hour_profile, target_hour_features)
from utils.saving import MODELS_DIR, ModelSaver, load_model

DATA_PATH = os.path.join("data", "btc_hourly.csv.gz")
CHECK_DAYS = 30                             # the check of section 2: origins of the last 30 days with 24 known hours after
RUN_TIMESTRAL = True                        # section 6 loads the Timestral weights (111 MB, Hugging Face cache)
BATCH = 256                                 # Timestral rows (origins x series and covariates) per forward pass (8 GB GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Colours and axes style shared by the plots (as in classical-models.ipynb)
BLUE, ORANGE, AQUA, GREY, INK, PURPLE = "#2a78d6", "#eb6834", "#1baf7a", "#b4b2a9", "#52514e", "#6250d6"


def tidy(ax, title=None):
    if title:
        ax.set_title(title, loc="left", fontsize=11)
    ax.grid(axis="y", color="#e5e4df", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

In [ ]:
# every model saved by utils/saving.py: one index.json per notebook
rows = []
for index in sorted(MODELS_DIR.glob("*/index.json")):
    for name, entry in json.loads(index.read_text(encoding="utf-8")).items():
        rows.append({"notebook": index.parent.name, "name": name, "file": entry["file"], "saved": entry["saved"],
                     "MB": round((index.parent / entry["file"]).stat().st_size / 1e6, 2)})
CATALOGUE = pd.DataFrame(rows, columns=["notebook", "name", "file", "saved", "MB"])
EXPORTS = sorted(p.parent for p in MODELS_DIR.glob("*/models.joblib"))      # the Timestral exports (section 6)
print(f"{len(CATALOGUE)} saved models in {MODELS_DIR}, Timestral exports: {[p.name for p in EXPORTS]}")
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 200):
    display(CATALOGUE.groupby("notebook")["name"].count().rename("models").to_frame())
    display(CATALOGUE)


def find(notebook, pattern):
    # The name of the one model of `notebook` whose name matches the regular expression `pattern` (None if missing)
    names = CATALOGUE.loc[CATALOGUE["notebook"] == notebook, "name"]
    hits = [n for n in names if re.search(pattern, n)]
    if len(hits) > 1:
        raise ValueError(f"{pattern!r} matches {len(hits)} models of {notebook}: {hits}")
    if not hits:
        print(f"not found in {notebook}: {pattern!r} (run the notebook with SAVE_MODELS = True)")
    return hits[0] if hits else None


def setting(name):
    # The setting written in a model's name, "... {'hidden_size': 64}" -> {"hidden_size": 64}
    return ast.literal_eval(re.search(r"\{.*\}", name).group(0))


def info(notebook, name):
    # The extra information saved with a model (epochs, seeds, target_scale...)
    return json.loads((MODELS_DIR / notebook / "index.json").read_text(encoding="utf-8"))[name]

## 2. Data and inputs

The saved models take the inputs of the notebooks that trained them, so they are rebuilt here the same way: the base data from `DATA_START` with `Return` in percent, the feature groups of `utils/hourly.py` and `utils/features.py`, and the **long format** (24 rows per origin, one per predicted hour).

Two sets of forecast origins:
- `LIVE`: the last hour of the data. Its 24 predicted hours are in the future: the features of a predicted hour are either calendar facts (hour, weekday, release calendar) or moves at least 24 hours old, so they are all known at the origin.
- `CHECK`: the origins of the last `CHECK_DAYS` days whose 24 hours are known, to compare the forecasts with what happened. They are in the test block, outside the training data of the models refitted on train + validation (not of the models refitted on all the data, the "month after the data" ones).

In [ ]:
hourly = pd.read_csv(DATA_PATH, index_col="Timestamp", parse_dates=True).loc[DATA_START:].copy()
hourly["Return"] *= U.PCT
assert (hourly.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all(), "the hourly grid has holes"
ret, absr = hourly["Return"].values, hourly["Return"].abs().values
releases = load_releases()

FEAT = pd.concat([U.base_features(hourly), U.long_lookback_features(hourly), U.range_vol_features(hourly),
                  origin_features(hourly)[ORIGIN_FEATURES], next_day_release_features(hourly, releases)], axis=1)
PROF = recent_hour_profile(hourly)                                   # row t: hour-of-day profile of the 90 days up to t
week_abs = pd.Series(absr).rolling(168).mean().values               # last week's average |r| at each hour

LIVE = np.array([len(hourly) - 1])
CHECK = np.arange(len(hourly) - U.H_AHEAD - 24 * CHECK_DAYS, len(hourly) - U.H_AHEAD)
assert hourly.index[CHECK[0]] >= TEST_START
print(f"{len(hourly):,} hours   {hourly.index[0]} -> {hourly.index[-1]}")
print(f"LIVE origin {hourly.index[LIVE[0]]}: forecasts {hourly.index[-1] + pd.Timedelta(hours=1)} -> "
      f"{hourly.index[-1] + pd.Timedelta(hours=U.H_AHEAD)}")
print(f"CHECK: {len(CHECK)} origins {hourly.index[CHECK[0]]} -> {hourly.index[CHECK[-1]]}")

In [ ]:
H_AHEAD = np.arange(1, U.H_AHEAD + 1)


def Xw(pos, cols):
    # Wide format: one row per origin
    return FEAT[cols].values[pos]


def Xl(pos, cols, hcols, hour_profile):
    # Long format: 24 rows per origin (origin-major order), as Xl() of the model notebooks; cols = origin features,
    # hcols = per-hour features. The predicted hours may lie after the data (LIVE): only their timestamps are used,
    # with moves at least 24 hours old
    o, h_ = np.repeat(pos, 24), np.tile(H_AHEAD, len(pos))
    t = pd.DatetimeIndex(hourly.index[o] + pd.to_timedelta(h_, unit="h"))
    tgt = target_hour_features(t, releases)
    extra = {
        "h_ahead": h_,
        "tgt_hour_sin": np.sin(2 * np.pi * t.hour / 24), "tgt_hour_cos": np.cos(2 * np.pi * t.hour / 24),
        "tgt_dow_sin": np.sin(2 * np.pi * t.dayofweek / 7), "tgt_dow_cos": np.cos(2 * np.pi * t.dayofweek / 7),
        "tgt_weekend": (t.dayofweek >= 5).astype(float),
        "tgt_hour_profile": np.asarray(hour_profile)[t.hour],
        "absret_same_hour_1d": absr[o + h_ - 24], "absret_same_hour_7d": absr[o + h_ - 168],
        "ret_same_hour_1d": ret[o + h_ - 24],
        "tgt_hour_profile_recent": PROF[o, t.hour],                  # profile known at the origin, for the predicted hour
        **{c: tgt[c].values for c in TARGET_HOUR_FEATURES},
    }
    base = np.repeat(FEAT[cols].values[pos], 24, axis=0)
    return np.hstack([base, np.column_stack([extra[c] for c in hcols])]) if hcols else base


def actual(pos):
    # the 24 returns after each origin, % (origins, 24)
    return ret[np.asarray(pos)[:, None] + H_AHEAD]


def hours_of(pos):
    return hourly.index[pos[0]] + pd.to_timedelta(H_AHEAD, unit="h")


LIVE_ABS = {}                                                        # |r| forecasts of the LIVE origin, for section 7

## 3. Type 1: one scikit-learn model

A scikit-learn model (or pipeline) is a joblib file: `load_model(notebook, name)` gives the fitted object back, with its scaler or other steps inside if it is a pipeline. Only its **inputs** have to be rebuilt; the `inputs` entry of each notebook stores the column groups, the feature sets and the hour profile of the train block they need.

**(a) The 24 hourly returns, wide format** (`classical-models.ipynb`, section 3): one row per origin with the `BASE` columns, 24 outputs.

In [ ]:
CL = "classical-models"
cl_inputs = load_model(CL, "inputs") if find(CL, "^inputs$") else None
name = find(CL, r"^24 hourly returns, wide: Ridge ")
if name and cl_inputs:
    ridge = load_model(CL, name)
    print(f"{name}\n  {ridge}")
    live = ridge.predict(Xw(LIVE, cl_inputs["BASE"]))[0]
    display(pd.DataFrame({"forecast return %": live}, index=hours_of(LIVE)).T.round(4))
    chk = ridge.predict(Xw(CHECK, cl_inputs["BASE"]))
    print(f"check, last {CHECK_DAYS} days: MAE skill against the zero forecast "
          f"{1 - np.abs(actual(CHECK) - chk).mean() / np.abs(actual(CHECK)).mean():+.4f} (the notebook: close to 0)")

**(b) The size of each hourly move, long format** (`classical-models.ipynb`, section 6.5): Gradient Boosting on the columns of `CHOSEN_SET`. Two saved fits:
- `GB fitted once`: trained on the pool rows before the test block, so the check is out of sample;
- `... month after the data`: the chosen monthly refit trained on every row known at the end of the data, the one to use for the `LIVE` forecast.

In [ ]:
if cl_inputs:
    cols, hcols = cl_inputs["SETS"][cl_inputs["CHOSEN_SET"]]
    print(f"CHOSEN_SET: {cl_inputs['CHOSEN_SET']} ({len(cols)} origin columns, per-hour {hcols})")
    once, month = find(CL, r"^hourly \|r\|: GB fitted once"), find(CL, r"^hourly \|r\|: GB .*month after the data$")
    if once:
        gb = load_model(CL, once)
        pred = gb.predict(Xl(CHECK, cols, hcols, cl_inputs["HOUR_PROFILE"]))
        print(f"{once}: check R2 {U.r2(np.abs(actual(CHECK)).ravel(), pred):.4f}")
    if month:
        gb = load_model(CL, month)
        LIVE_ABS[f"classical: {month[12:]}"] = gb.predict(Xl(LIVE, cols, hcols, cl_inputs["HOUR_PROFILE"]))
        print(f"{month}: LIVE forecast of |r| (%), first 6 hours {LIVE_ABS[f'classical: {month[12:]}'][:6].round(3)}")

## 4. Type 2: dicts and lists of scikit-learn objects

Some entries hold several fitted objects in one file: a dict (the quantile models, one per level; K-Means with its scaler) or a list (the MLP of `neural-models.ipynb` §6, one per seed). They load the same way; then each item is used as in the notebook.

**(a) Quantile models** (`classical-models.ipynb`, section 7): `{0.1: model, 0.5: model, 0.9: model}` on the long format of `CHOSEN_SET`, for the return of each hour. These are the raw quantiles: the notebook's conformal calibration rescales them with the errors of the last 30 days, which is not a saved model.

In [ ]:
name = find(CL, r"^hourly ranges: GB quantile \(0\.1, 0\.5, 0\.9\), train \+ validation")
if name and cl_inputs:
    q_models = load_model(CL, name)
    print(f"{name}: {type(q_models).__name__} with keys {list(q_models)}")
    X_live = Xl(LIVE, cols, hcols, cl_inputs["HOUR_PROFILE"])
    q = np.sort(np.column_stack([q_models[t].predict(X_live) for t in sorted(q_models)]), axis=1)   # no crossing
    display(pd.DataFrame(q.T, index=["q10 %", "q50 %", "q90 %"], columns=hours_of(LIVE)).round(3))
    X_chk = Xl(CHECK, cols, hcols, cl_inputs["HOUR_PROFILE"])
    lo, hi = (q_models[t].predict(X_chk) for t in (0.1, 0.9))
    y = actual(CHECK).ravel()
    print(f"check: {((y >= np.minimum(lo, hi)) & (y <= np.maximum(lo, hi))).mean():.1%} of the hours inside the raw 80% range")

**(b) K-Means day clusters** (`data-analysis.ipynb`, section 6.1): `{"columns", "scaler", "kmeans"}`. The days are described at 23:00 by three columns of that section; the scaler standardizes them with the train block, K-Means gives the cluster.

In [ ]:
DA = "data-analysis"
name = find(DA, r"^K-Means day clusters")
if name:
    km = load_model(DA, name)
    r, close, volume = hourly["Return"], hourly["Close"], hourly["Volume"].clip(lower=0)
    day_cols = pd.DataFrame({"ret_1d": (close / close.shift(24) - 1) * 100,            # as in data-analysis.ipynb, 6
                             "vol_1d": r.rolling(24).std(),
                             "volume_ratio": np.log1p(volume.rolling(24).mean()) - np.log1p(volume.rolling(168).mean())})
    last_days = day_cols[day_cols.index.hour == 23][km["columns"]].tail(10)
    centres = pd.DataFrame(km["scaler"].inverse_transform(km["kmeans"].cluster_centers_), columns=km["columns"])
    print("cluster centres (train block):")
    display(centres.round(3))
    print("the last 10 days:")
    display(last_days.assign(cluster=km["kmeans"].predict(km["scaler"].transform(last_days))).round(3))

**(c) A list of models, one per seed** (`neural-models.ipynb`, section 6): the scikit-learn MLP for next-day volatility was refitted with 5 seeds; the forecast is the average of the 5. Input: one row per origin with the columns `DV_COLS`; target: the log of the next day's realized volatility.

In [ ]:
NN = "neural-models"
nn_inputs = load_model(NN, "inputs") if find(NN, "^inputs$") else None
name = find(NN, r"^next-day volatility: MLP ")
if name and nn_inputs:
    mlps = load_model(NN, name)
    print(f"{name}: {len(mlps)} pipelines ({' -> '.join(type(s).__name__ for s in mlps[0].named_steps.values())})")
    live = np.mean([m.predict(Xw(LIVE, nn_inputs["DV_COLS"])) for m in mlps], axis=0)
    print(f"LIVE forecast of the next day's realized volatility: {np.exp(live[0]):.2f}%")
    chk = np.mean([m.predict(Xw(CHECK, nn_inputs["DV_COLS"])) for m in mlps], axis=0)
    print(f"check R2 on the log volatility: {U.r2(U.day_log_vol(actual(CHECK)), chk):.4f}")

## 5. Type 3: PyTorch networks

The networks are saved as **state_dicts** (the weights), one per seed, on the CPU. A state_dict has no code: the network is built again with its class and its setting, then the weights are loaded. So this section needs:
- **the class**, exactly as in `neural-models.ipynb` (section 2): copied below for `MLPNet` and `RecurrentNet`;
- **the setting**: in the name (`{'hidden': (256, 128), ...}`) and the fixed options of the notebook (dropout, number of layers);
- **the scalings**: the inputs are standardized with the train block (`sc_abs`, `SEQ_MU`, `SEQ_SD` of the `inputs` entry) and the networks predict the target divided by `target_scale` (saved in `index.json`).

The forecast is the average of the seeds, as in the notebook.

In [ ]:
# the classes of neural-models.ipynb, section 2 (a state_dict only loads into the same architecture)
class MLPNet(nn.Module):
    # Fully connected network: (Linear -> ReLU -> Dropout) per hidden layer, then a linear output
    def __init__(self, n_in, hidden=(64, 32), n_out=1, dropout=0.0):
        super().__init__()
        layers, d = [], n_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.net = nn.Sequential(*layers, nn.Linear(d, n_out))
        self.n_out = n_out

    def forward(self, x):
        out = self.net(x)
        return out[:, 0] if self.n_out == 1 else out


class RecurrentNet(nn.Module):
    # model_test.ipynb: a stacked RNN/LSTM reads the sequence, a linear head maps the last hidden state to the outputs
    def __init__(self, cell, n_features, hidden_size=64, num_layers=2, dropout=0.2, n_out=24):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM}[cell]
        self.rnn = rnn_cls(n_features, hidden_size, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, n_out))
        self.n_out = n_out

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.head(out[:, -1])
        return out[:, 0] if self.n_out == 1 else out


@torch.no_grad()
def seeds_forecast(make_net, states, X, scale, batch=1024):
    # average forecast of the networks of every seed, back in the unit of the target
    preds = []
    for state in states:
        net = make_net()
        net.load_state_dict(state)
        net.to(DEVICE).eval()
        preds.append(np.concatenate([net(torch.from_numpy(X[i:i + batch]).to(DEVICE)).cpu().numpy()
                                     for i in range(0, len(X), batch)]))
    return np.mean(preds, axis=0) * scale

**(a) MLP, long format** (section 5.2): input the columns of `CHOSEN_SET` standardized by `sc_abs`, one output (the |r| of the row's hour), dropout 0.1.

In [ ]:
name = find(NN, r"^hourly \|r\|: MLP, long format \{")
if name and nn_inputs:
    states, p, scale = load_model(NN, name), setting(name), info(NN, name)["target_scale"]
    to_x = lambda pos: nn_inputs["sc_abs"].transform(
        Xl(pos, nn_inputs["COLS"], nn_inputs["HCOLS"], nn_inputs["HOUR_PROFILE"])).astype(np.float32)
    make = lambda: MLPNet(len(nn_inputs["COLS"]) + len(nn_inputs["HCOLS"]), p["hidden"], n_out=1, dropout=0.1)
    print(f"{name}: {len(states)} seeds, {info(NN, name)['epochs']} epochs, target scale {scale:.4f}")
    print(f"check R2 {U.r2(np.abs(actual(CHECK)).ravel(), seeds_forecast(make, states, to_x(CHECK), scale)):.4f}")
    LIVE_ABS[f"neural: {name[12:]}"] = seeds_forecast(make, states, to_x(LIVE), scale)

**(b) LSTM on the 168-hour sequence** (section 5.3): input the 168 hours before the origin, 10 channels normalized per window (`raw_seq` of section 1.4), then standardized with `SEQ_MU`, `SEQ_SD` and clipped at ±10; 24 outputs.

The other networks of section 5.3 load the same way, with their class and input: `FutureHeadLSTM` and `VolumeAttentionLSTM` read the sequence followed by the standardized features `FUT` of the 24 predicted hours (`seq_fut`, with `FUT_MU`, `FUT_SD`); the scale-free variants divide the target by last week's average |r| (`week_abs` at the origin) and their sequence uses `SF_MU`, `SF_SD`. Copy their class and input function from the notebook.

In [ ]:
hl_pct = ((hourly["High"] - hourly["Low"]) / hourly["Close"]).values * U.PCT
log_close = np.log(hourly["Close"].values)
log_vol = np.log1p(hourly["Volume"].clip(lower=0).values)
filled = hourly["filled_minutes"].values.astype(float)
calendar = np.column_stack([np.sin(2 * np.pi * hourly.index.hour / 24), np.cos(2 * np.pi * hourly.index.hour / 24),
                            np.sin(2 * np.pi * hourly.index.dayofweek / 7), np.cos(2 * np.pi * hourly.index.dayofweek / 7)])


def seq(pos, inputs):
    # raw_seq() of neural-models.ipynb, section 1.4, standardized with the train block: (origins, SEQ_HOURS, channels)
    n = inputs["SEQ_HOURS"]
    idx = pos[:, None] + np.arange(-n + 1, 1)
    x = np.empty(idx.shape + (len(inputs["CHANNELS"]),), np.float32)
    x[..., 0], x[..., 1], x[..., 2] = ret[idx], absr[idx], hl_pct[idx]
    x[..., 3] = (log_close[idx] - log_close[pos][:, None]) * U.PCT
    x[..., 4] = log_vol[idx] - log_vol[idx].mean(axis=1, keepdims=True)
    x[..., 5] = filled[idx]
    x[..., 6:] = calendar[idx]
    mu, sd = inputs["SEQ_MU, SEQ_SD"]
    return np.clip((x - mu) / sd, -10, 10).astype(np.float32)


name = find(NN, r"^hourly \|r\|: LSTM \{")
if name and nn_inputs:
    states, p, scale = load_model(NN, name), setting(name), info(NN, name)["target_scale"]
    make = lambda: RecurrentNet("lstm", len(nn_inputs["CHANNELS"]), hidden_size=p["hidden_size"], num_layers=2, dropout=0.2, n_out=24)
    print(f"{name}: {len(states)} seeds, {info(NN, name)['epochs']} epochs")
    print(f"check R2 {U.r2(np.abs(actual(CHECK)).ravel(), seeds_forecast(make, states, seq(CHECK, nn_inputs), scale).ravel()):.4f}")
    LIVE_ABS[f"neural: {name[12:]}"] = seeds_forecast(make, states, seq(LIVE, nn_inputs), scale)[0]

## 6. Type 4: the Timestral export

`timestral-zero-shot.ipynb` (section 3.6) exports the size-of-the-move models to `models/<id>/`, where `<id>` is the name of the model chosen on validation:
- `models.joblib`: `{"best": name, "models": {name: model}}`. A model is a fitted Gradient Boosting (long format, columns `gb columns` of the metadata), the **setting** of Timestral (a dict: repository, revision, series, context, covariates, rescaling), or an ensemble `{"average of": [members]}`;
- `metadata.json`: the columns, the hour profile of the train block, the scores and the period of the data.

Timestral is used zero-shot, so its "model" is its setting: the weights come from Hugging Face (`TimestralForecaster.from_pretrained`), and the forecast is the mean |r| of its quantiles, rescaled by `a + b x` when the export says so.

In [ ]:
if EXPORTS:
    export = EXPORTS[-1]
    meta = json.loads((export / "metadata.json").read_text(encoding="utf-8"))
    bundle = joblib.load(export / "models.joblib")
    print(f"{export}: best {bundle['best']!r}, exported {meta['exported']}, data {meta['data'][0]} -> {meta['data'][1]}")
    display(pd.DataFrame({k: {"kind": type(v).__name__ if not isinstance(v, dict) else
                              ("ensemble" if "average of" in v else "Timestral setting"), **meta["scores"].get(k, {})}
                          for k, v in bundle["models"].items()}).T)
else:
    print("no Timestral export in models/: run timestral-zero-shot.ipynb")

In [ ]:
FC = {}


def forecaster(spec):
    # the Timestral model of the export, downloaded to the Hugging Face cache on first use
    from timestral import TimestralForecaster
    key = (spec["repo"], spec["revision"])
    if key not in FC:
        FC[key] = TimestralForecaster.from_pretrained(spec["repo"], device=DEVICE, revision=spec["revision"])
    return FC[key]


def timestral_abs(pos, spec):
    # Mean |r| of the 24 hours after each origin from Timestral's quantiles, as timestral-zero-shot.ipynb, section 2
    fc = forecaster(spec)
    ctx = spec["context hours"]
    assert pos.min() >= ctx - 1, "not enough history for the context"
    ext = hourly.index.append(pd.date_range(hourly.index[-1], periods=U.H_AHEAD + 1, freq="h")[1:])   # + 24 future hours
    hour, dow = np.asarray(ext.hour), np.asarray(ext.dayofweek)
    pad = lambda a: np.r_[np.asarray(a, dtype=float), np.full(U.H_AHEAD, np.nan)]       # past-only rows: no future
    groups = {"clock, categorical": ("categorical_known_covariates", np.vstack([hour, dow]).astype(float)),
              "clock, numeric": ("known_covariates", np.vstack([np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24),
                                                                (dow >= 5).astype(float)])),
              "releases": ("known_covariates", target_hour_features(ext, releases)[TARGET_HOUR_FEATURES].values.T.astype(float)),
              "range, volume": ("past_covariates", np.vstack([pad(hl_pct), pad(log_vol)]))}
    series = {"r": ret, "|r|": absr}[spec["series"]]
    step = max(1, BATCH // (1 + sum(len(groups[g][1]) for g in spec["covariates"])))   # origins per pass: BATCH rows
    q = []
    for i in range(0, len(pos), step):
        ctx_idx = pos[i:i + step, None] + np.arange(-ctx + 1, 1)          # the ctx hours up to the origin
        all_idx = pos[i:i + step, None] + np.arange(-ctx + 1, U.H_AHEAD + 1)   # the same and the 24 predicted hours
        kw = {}
        for g in spec["covariates"]:
            arg, rows = groups[g]
            x = rows[:, ctx_idx if arg == "past_covariates" else all_idx].transpose(1, 0, 2)
            kw[arg] = x if arg not in kw else np.concatenate([kw[arg], x], axis=1)
        kw = {k: v.astype(np.float64 if k.startswith("categorical") else np.float32) for k, v in kw.items()}
        f = fc.forecast(series[ctx_idx].astype(np.float32), prediction_length=U.H_AHEAD, **kw)
        q.append(f.quantiles[:, 0].float().cpu().numpy())
    q = np.concatenate(q)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    lv = np.r_[0.0, fc.quantile_levels, 1.0]                         # trapezoid weights of the quantiles, flat tails
    w = (lv[2:] - lv[:-2]) / 2
    w[0], w[-1] = w[0] + lv[1] / 2, w[-1] + (1 - lv[-2]) / 2
    e = (np.abs(q) @ w).reshape(-1)
    if spec["rescaling a, b"]:
        a, b = spec["rescaling a, b"]
        e = a + b * e
    return e


def export_forecast(name, pos):
    # forecast of |r| (long format) of one model of the export: Gradient Boosting, Timestral or an ensemble
    m = bundle["models"][name]
    if isinstance(m, dict) and "average of" in m:
        return np.mean([export_forecast(k, pos) for k in m["average of"]], axis=0)
    if isinstance(m, dict):
        return timestral_abs(pos, m)
    gb_cols = meta["gb columns"]                                     # origin columns, then the per-hour columns
    hcols = [c for c in gb_cols if c in ("h_ahead", "tgt_hour_sin", "tgt_hour_cos", "tgt_dow_sin", "tgt_dow_cos",
                                          "tgt_weekend", "tgt_hour_profile", "absret_same_hour_1d", "absret_same_hour_7d",
                                          "ret_same_hour_1d", "tgt_hour_profile_recent", *TARGET_HOUR_FEATURES)]
    cols = gb_cols[:len(gb_cols) - len(hcols)]
    assert cols + hcols == gb_cols
    return m.predict(Xl(pos, cols, hcols, meta["hour profile"]))


if EXPORTS and RUN_TIMESTRAL:
    best = bundle["best"]
    chk = export_forecast(best, CHECK)
    print(f"{best}: check R2 {U.r2(np.abs(actual(CHECK)).ravel(), chk):.4f}")
    LIVE_ABS[f"Timestral export: {best}"] = export_forecast(best, LIVE)

## 7. The LIVE forecasts side by side

The |r| forecasts of the 24 hours after the last hour of the data, from every model loaded above. They should agree on the shape of the day (the hour-of-day profile) and differ mostly in the level.

In [ ]:
if LIVE_ABS:
    t = hours_of(LIVE)
    live_tab = pd.DataFrame(LIVE_ABS, index=t)
    fig, ax = plt.subplots(figsize=(12, 4))
    for (k, v), c in zip(live_tab.items(), [BLUE, ORANGE, AQUA, PURPLE, INK]):
        ax.plot(t, v, "o-", ms=3, color=c, label=k[:70])
    ax.plot(t, week_abs[LIVE[0]] * np.ones(len(t)), color=GREY, ls="--", lw=0.8, label="last week's average |r|")
    ax.set_ylabel("|r| (%)")
    ax.legend(frameon=False, fontsize=8)
    tidy(ax, f"Forecast size of each hourly move after {hourly.index[LIVE[0]]}")
    plt.show()
    display(live_tab.T.round(3))

## 8. Notes

- **Same inputs as in training.** A loaded model gives wrong forecasts, without any error, if its columns come in another order or are built differently. The functions above copy those of the notebooks; the `inputs` entries keep the column lists, the feature sets chosen on validation and the train scalings, so they are not recomputed here.
- **Which fit to use.** The fits "for the test block" were trained on train + validation (up to mid-2024). For a forecast after the data, prefer the "month after the data" refits (trained on every known row, the newer rows weighted more) or rerun the notebook on newer data.
- **Release correction and conformal ranges** (`classical-models.ipynb` §6.5, §7) are computed from recent errors, not saved: apply them as in the notebook.
- **PyTorch versions.** The state_dicts load in any recent PyTorch; the class must match the one of the training run (same layers and names).
- **Loading a single file.** `load_model(notebook, name)` reads `index.json` for the file name; `ModelSaver(notebook).names()` lists the names.